In [1]:
import os
import pandas as pd
from icalendar import Calendar, Event
from datetime import datetime
import pytz
import re

group = "ZICSS1-1211"
df = pd.read_csv(f"schedules/{group}.csv")

cal = Calendar()
cal.add('prodid', '-//UEK Schedule//PL')
cal.add('version', '2.0')

tz = pytz.timezone('Europe/Warsaw')

for _, row in df.iterrows():
    try:
        # "Pn 13:15 - 15:45 (3g.)" formatını parse et
        time_str = row['Dzień, godzina']
        times = re.findall(r'\d{2}:\d{2}', time_str)
        if len(times) < 2:
            continue
        start_t, end_t = times[0], times[1]

        date = datetime.strptime(str(row['Termin']), '%Y-%m-%d')
        start = tz.localize(date.replace(
            hour=int(start_t.split(':')[0]),
            minute=int(start_t.split(':')[1])
        ))
        end = tz.localize(date.replace(
            hour=int(end_t.split(':')[0]),
            minute=int(end_t.split(':')[1])
        ))

        event = Event()
        event.add('summary', f"{row['Przedmiot']} ({row['Typ']})")
        event.add('dtstart', start)
        event.add('dtend', end)
        event.add('location', str(row['Sala']))
        event.add('description', f"Nauczyciel: {row['Nauczyciel']}")
        cal.add_component(event)
    except Exception as e:
        print(f"Satır atlandı: {e}")

os.makedirs("calendars", exist_ok=True)
with open(f"calendars/{group}.ics", 'wb') as f:
    f.write(cal.to_ical())

print("Kaydedildi!")

KeyError: 'Start time'